Create a supervisor tool which has a research tool, a python code tool

In [3]:
pip install -U langchain-tavily

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph_supervisor import create_supervisor
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
from langgraph.graph import START, END, StateGraph
import os
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

In [6]:
load_dotenv()
if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError("GROQ_API_KEY not found in environment variables.")
llm = ChatGroq(model="openai/gpt-oss-120b")

In [11]:
web_search = TavilySearch(
    max_results=5
)

In [12]:
research_agent = create_agent(
    llm,
    tools=[web_search],
    name="research_agent",
    system_prompt="""
    You are a research expert.
    Always search before answering.
    """
)

In [14]:
result = web_search.invoke({
    "query": "Latest Python programming language version"
})

print(result)

{'query': 'Latest Python programming language version', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.quora.com/Which-is-the-latest-python-version', 'title': 'Which is the latest python version?', 'content': 'Which is the latest python version? Image 1: Profile photo for Assistant. As of 1 September 2025, the latest stable Python release is Python 3.13.1 (released 2025-08-25). *   Python 3.13 is the current major series; 3.13.1 is the most recent patch release. Latest version of Python programming language is 3.7.0.Its stable version was released on 27 June. Python 3.7, the latest version of the language aimed at making complex tasks simple, is now in production release. You can even learn more about what’s new in python 3.7.0 in the article by going through the link given below:. Which is the latest version of the Python programming language, and when is it going to be updated? Python 3.3.3 is latest version. Originally Answered: Which is 

In [15]:
python_agent = create_agent(
    llm,
    name="python_agent",
    system_prompt="""
        You are a python coding assisstant.
        Help the user write proper code when asked. 
        In the end also give explanation to code you write"""
)

In [20]:
res2 = python_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Write Python code to double an input number."
        }
    ]
})

In [21]:
for msg in res2["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Write Python code to double an input number.
================================== Ai Message ==================================
Name: python_agent

**Python code**

```python
def double_number(x: float) -> float:
    """
    Return the value of x multiplied by 2.

    Parameters
    ----------
    x : float
        The number you want to double.  It can be an int or a float.

    Returns
    -------
    float
        The doubled value.
    """
    return x * 2


if __name__ == "__main__":
    # Ask the user for a number
    try:
        user_input = input("Enter a number to double: ")
        # Convert the input string to a float (works for ints as well)
        number = float(user_input)
    except ValueError:
        print("That doesn't look like a valid number.")
    else:
        result = double_number(number)
        print(f"The double of {number} is {result}.")
```

**Explanation**

1. **`double_numbe

# Create supervisor

In [22]:
from langgraph_supervisor import create_supervisor
from langgraph.checkpoint.memory import InMemorySaver

In [23]:
workflow = create_supervisor(
    agents = [research_agent, python_agent],
    model=llm,
    prompt="""
        You supervise two specialist agents.

        - research_agent:
            Use for questions requiring web research or current information.

        - python_agent:
            Use for writing, debugging, or executing Python code.

        Assign one agent at a time.
        Do not answer the user's question yourself.
        Continue delegating until the task is complete.
        """
)

In [24]:
app = workflow.compile(
    checkpointer=InMemorySaver()
)

In [25]:
config1={
    "configurable":{
        "thread_id": "thread-1"
    }
}

In [28]:
result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Find the year LLM was created and then write Python code that squares that year."
            }
        ]
    },
    config1
)

BadRequestError: Error code: 400 - {'error': {'message': "Tool call validation failed: tool call validation failed: parameters for tool tavily_search did not match schema: errors: [missing properties: 'query']", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "tavily_search", "arguments": {"cursor": 0, "id": 0}}'}}

In [ ]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Find the year Python was created and then write Python code that doubles that year.
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_research_agent (fc_6428c5a9-5f33-413e-891f-1e68aa81def1)
 Call ID: fc_6428c5a9-5f33-413e-891f-1e68aa81def1
  Args:
    : {}
================================= Tool Message =================================
Name: transfer_to_research_agent

Successfully transferred to research_agent
================================== Ai Message ==================================
Name: research_agent

**Year Python was created:** 1991【2†results.0.title】  

**Python code that doubles that year**

```python
# Year Python was first released
python_year = 1991

# Double the year
double_year = python_year * 2

print("The year Python was created:", python_year)
print("Double that year:", double_year)
```

**Output**

```
The y